**0. Imports**

In [ ]:
import xarray as xr
import numpy as np
from pathlib import Path
from datetime import datetime
import os

**1. Config**

In [ ]:
# Must match the date range and grid resolution used in the processing notebooks
START_DATE   = "2025-12-01"
END_DATE     = "2025-12-31"
GRID_RES_DEG = 1.0

# Output roots — must match what each processing notebook used
ATC_OUTPUT_ROOT  = Path("/usr/people/raucher/Documents/Coding1/KNMI/KNMI/ATC_output")   # ATL_TC, EBD, ICE
ACTC_OUTPUT_ROOT = Path("/usr/people/raucher/Documents/Coding1/KNMI/KNMI/ACTC_output")  # AC_TC

start_dt = datetime.strptime(START_DATE, "%Y-%m-%d").date()
end_dt   = datetime.strptime(END_DATE,   "%Y-%m-%d").date()
date_tag = f"{start_dt:%Y%m%d}_{end_dt:%Y%m%d}"
deg_tag  = f"{GRID_RES_DEG}deg"

in_dir_atc  = ATC_OUTPUT_ROOT  / date_tag
in_dir_actc = ACTC_OUTPUT_ROOT / date_tag
out_dir     = ATC_OUTPUT_ROOT  / date_tag   # combined files go here too
os.makedirs(out_dir, exist_ok=True)

print(f"Date range:  {date_tag}")
print(f"ATL inputs:  {in_dir_atc}")
print(f"AC  inputs:  {in_dir_actc}")
print(f"Output dir:  {out_dir}")

**2. Check input files exist**

In [ ]:
files = {
    "atl_tc_3d":      in_dir_atc  / f"ATL_TC_2A_{deg_tag}_{date_tag}_occurrence_3d.nc",
    "atl_tc_lh":      in_dir_atc  / f"ATL_TC_2A_{deg_tag}_{date_tag}_occurrence_latheight.nc",
    "ac_tc_3d":       in_dir_actc / f"AC_TC_2B_{deg_tag}_{date_tag}_occurrence_3d.nc",
    "ac_tc_lh":       in_dir_actc / f"AC_TC_2B_{deg_tag}_{date_tag}_occurrence_latheight.nc",
    "ebd_3d":         in_dir_atc  / f"ATL_EBD_2A_{deg_tag}_{date_tag}_3d.nc",
    "ebd_lh":         in_dir_atc  / f"ATL_EBD_2A_{deg_tag}_{date_tag}_latheight.nc",
    "ice_3d":         in_dir_atc  / f"ATL_ICE_2A_{deg_tag}_{date_tag}_3d.nc",
    "ice_lh":         in_dir_atc  / f"ATL_ICE_2A_{deg_tag}_{date_tag}_latheight.nc",
}

all_ok = True
for key, path in files.items():
    status = "OK" if path.exists() else "MISSING"
    if status == "MISSING":
        all_ok = False
    print(f"  {status}  {key:15s}  {path.name}")

if not all_ok:
    raise FileNotFoundError("One or more input files are missing — re-run the relevant processing notebooks first.")
print("\nAll input files found.")

**3. Combine lat/height (zonal mean) files**

In [ ]:
# ATL_TC and AC_TC both have variables named 'occurrence', 'ice_count', 'total_count'
# — prefix them to avoid conflicts. ATL_TC also has 'temperature' and 'temp_count'.
ds_atl_tc_lh = xr.open_dataset(files["atl_tc_lh"]).rename(
    {v: f"atl_tc_{v}" for v in ["occurrence", "ice_count", "total_count", "temperature", "temp_count"]})

ds_ac_tc_lh  = xr.open_dataset(files["ac_tc_lh"]).rename(
    {v: f"ac_tc_{v}"  for v in ["occurrence", "ice_count", "total_count"]})

ds_ebd_lh = xr.open_dataset(files["ebd_lh"])
ds_ice_lh = xr.open_dataset(files["ice_lh"])

ds_combined_lh = xr.merge([ds_atl_tc_lh, ds_ac_tc_lh, ds_ebd_lh, ds_ice_lh])

out_lh = out_dir / f"combined_{deg_tag}_{date_tag}_latheight.nc"
ds_combined_lh.to_netcdf(out_lh)

print(f"Saved: {out_lh.name}")
print(f"Dims:  {dict(ds_combined_lh.dims)}")
print(f"Vars ({len(ds_combined_lh.data_vars)}):")
for v in ds_combined_lh.data_vars:
    print(f"  {v}")

**4. Combine 3D (lat × lon × height) files**

In [ ]:
ds_atl_tc_3d = xr.open_dataset(files["atl_tc_3d"]).rename(
    {v: f"atl_tc_{v}" for v in ["occurrence", "ice_count", "total_count", "temperature", "temp_count"]})

ds_ac_tc_3d  = xr.open_dataset(files["ac_tc_3d"]).rename(
    {v: f"ac_tc_{v}"  for v in ["occurrence", "ice_count", "total_count"]})

ds_ebd_3d = xr.open_dataset(files["ebd_3d"])
ds_ice_3d = xr.open_dataset(files["ice_3d"])

ds_combined_3d = xr.merge([ds_atl_tc_3d, ds_ac_tc_3d, ds_ebd_3d, ds_ice_3d])

out_3d = out_dir / f"combined_{deg_tag}_{date_tag}_3d.nc"
ds_combined_3d.to_netcdf(out_3d)

print(f"Saved: {out_3d.name}")
print(f"Dims:  {dict(ds_combined_3d.dims)}")
print(f"Vars ({len(ds_combined_3d.data_vars)}):")
for v in ds_combined_3d.data_vars:
    print(f"  {v}")

**5. Compute AC_TC minus ATL_TC occurrence difference**

In [ ]:
# Positive values: AC_TC detects more ice than ATL_TC lidar alone (synergy adds value)
# Negative values: ATL_TC detects more (rare, indicates disagreement)

# Lat/height
ds_combined_lh["occurrence_diff"] = (
    ds_combined_lh["ac_tc_occurrence"] - ds_combined_lh["atl_tc_occurrence"]
)
ds_combined_lh["occurrence_diff"].attrs["long_name"] = "AC_TC minus ATL_TC ice occurrence"
ds_combined_lh["occurrence_diff"].attrs["note"] = "Positive = synergy product sees more ice than lidar alone"
ds_combined_lh.to_netcdf(out_lh, mode="w")

# 3D
ds_combined_3d["occurrence_diff"] = (
    ds_combined_3d["ac_tc_occurrence"] - ds_combined_3d["atl_tc_occurrence"]
)
ds_combined_3d["occurrence_diff"].attrs["long_name"] = "AC_TC minus ATL_TC ice occurrence"
ds_combined_3d["occurrence_diff"].attrs["note"] = "Positive = synergy product sees more ice than lidar alone"
ds_combined_3d.to_netcdf(out_3d, mode="w")

print("Difference variable 'occurrence_diff' added and files re-saved.")
print(f"  latheight range: {float(ds_combined_lh['occurrence_diff'].min()):.3f} .. {float(ds_combined_lh['occurrence_diff'].max()):.3f}")
print(f"  3d range:        {float(ds_combined_3d['occurrence_diff'].min()):.3f} .. {float(ds_combined_3d['occurrence_diff'].max()):.3f}")

**5. Quick sanity check**

In [ ]:
print("=== Combined lat/height ===")
print(ds_combined_lh)
print("\n=== Combined 3D ===")
print(ds_combined_3d)